# Semantische Varianz-Analyse von Synthetischen Daten 📊

In der Phase der **Data Curation** ist es entscheidend zu wissen, ob deine generierten synthetischen Dialoge (z. B. aus der Data Generation) eine hohe inhaltliche Vielfalt aufweisen oder ob sich das LLM zu oft wiederholt.

Dieses Notebook hilft dir dabei:
* Deine JSONL-Transkripte einzulesen.
* Die Texte mithilfe von Sentence-Transformers (`all-MiniLM-L6-v2`) in numerische Vektoren (Embeddings) zu übersetzen.
* Über die **Kosinus-Ähnlichkeit (Cosine Similarity)** mathematisch zu berechnen, wie ähnlich sich deine Dialoge untereinander sind.
* Eine visuelle Verteilung als Histogramm auszugeben.

## 1. Benötigte Bibliotheken importieren

Wir starten damit, alle notwendigen Python-Pakete zu laden:
* `json` & `numpy`: Zum Verarbeiten der Datei und für mathematische Berechnungen.
* `sklearn.metrics.pairwise`: Enthält die Funktion `cosine_similarity`, um die Ähnlichkeit der Vektoren zu bestimmen.
* `sentence_transformers`: Das HuggingFace-Framework, das unser Embedding-Modell auf der GPU ausführt.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

print("✅ Bibliotheken erfolgreich geladen!")

## 2. Kernfunktion zur Varianz-Analyse

Die folgende Funktion `analyze_variance` erledigt die Hauptarbeit:
1. **Zeilenweises Einlesen:** Sie öffnet die angegebene JSONL-Datei und fischt sich jeden Text heraus, der unter dem Schlüssel `text` gespeichert ist.
2. **GPU-Beschleunigung:** Das Modell `all-MiniLM-L6-v2` wird explizit auf `device='cuda'` geladen, damit die Grafikkarte die rechenintensive Vektor-Erstellung übernimmt.
3. **Ähnlichkeits-Matrix:** Wir berechnen die Kreuz-Ähnlichkeit aller Texte. Da ein Text im Vergleich zu sich selbst immer den Wert `1.0` liefert, setzen wir die Hauptdiagonale der Matrix per `np.nan` aus, um das Ergebnis nicht zu verfälschen.

In [ ]:
def analyze_variance(file_path: str):
    print(f'Lade Datensatz aus: {file_path}...')
    texts = []

    # Schritt 1: JSONL-Datei parsen
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                if 'text' in data:
                    texts.append(data['text'])

    if not texts:
        print('❌ Fehler: Keine Texte im Feld "text" gefunden!')
        return None, None

    print(f'✅ {len(texts)} Dialoge erfolgreich geladen. Starte Embedding-Berechnung auf der GPU...')

    # Schritt 2: Embeddings erzeugen
    model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

    # Schritt 3: Kosinus-Ähnlichkeit berechnen
    print('Berechne semantische Ähnlichkeitsmatrix...')
    sim_matrix = cosine_similarity(embeddings)
    
    # Selbstvergleiche (Wert 1.0 auf der Diagonale) herausfiltern
    np.fill_diagonal(sim_matrix, np.nan)

    # Metriken ermitteln
    avg_sim = np.nanmean(sim_matrix)
    max_sim = np.nanmax(sim_matrix)
    min_sim = np.nanmin(sim_matrix)

    # Schritt 4: Report ausgeben
    print('\n' + '='*50)
    print('       ERGEBNIS DER SEMANTISCHEN VARIANZ-ANALYSE       ')
    print('='*50)
    print(f'Analysierte Datensätze     : {len(texts)}')
    print(f'Durchschnittl. Ähnlichkeit: {avg_sim:.4f}  (Je niedriger, desto besser die Vielfalt)')
    print(f'Höchste Ähnlichkeit        : {max_sim:.4f}  (Achtung bei Werten nahe 1.0: Duplikat-Gefahr!)')
    print(f'Geringste Ähnlichkeit      : {min_sim:.4f}  (Maximale inhaltliche Bandbreite)')
    print('='*50)

    print('\nAutomatisierte Interpretation:')
    if avg_sim < 0.65:
        print('🌟 Hervorragend: Dein Datensatz besitzt eine exzellente semantische Vielfalt!')
    elif avg_sim < 0.80:
        print('👍 Solider Wert: Gute Balance aus thematischem Bezug und Abwechslung.')
    else:
        print('⚠️ Achtung: Die Dialoge ähneln sich thematisch stark. Erhöhe die Varianz deiner Prompts!')
        
    return sim_matrix, texts

## 3. Ausführung auf deinen Datensatz

Trage hier den korrekten Pfad zu deiner generierten `.jsonl`-Datei ein und führe die Zelle aus. Achte darauf, dass dein GPU-Container läuft.

In [ ]:
# Passe den Dateipfad bei Bedarf an
FILE_PATH = '/data/nemo-fraud-detection/notebooks/01_Data_Generation/data/transcripts.jsonl'

# Funktion aufrufen
sim_matrix, texts = analyze_variance(FILE_PATH)

## 4. Visualisierung der Ähnlichkeitsverteilung

Ein reiner Mittelwert reicht oft nicht aus. Das folgende Histogramm zeigt dir grafisch, wie sich die Ähnlichkeitswerte über alle Textpaare hinweg verteilen. 
* **Linker Bereich (niedrige Werte):** Völlig unterschiedliche Szenarien.
* **Rechter Bereich (hohe Werte, nahe 1.0):** Potenzielle Duplikate oder redundante Daten, die du vor dem Finetuning bereinigen solltest.

In [ ]:
if sim_matrix is not None:
    # Alle NaN-Werte für das Diagramm ignorieren
    flat_sims = sim_matrix[~np.isnan(sim_matrix)]

    plt.figure(figsize=(10, 5))
    sns.histplot(flat_sims, bins=50, kde=True, color='teal')
    plt.title('Verteilung der semantischen Ähnlichkeiten im Datensatz', fontsize=12, fontweight='bold')
    plt.xlabel('Kosinus-Ähnlichkeit (Cosine Similarity)', fontsize=10)
    plt.ylabel('Anzahl der Textpaare', fontsize=10)
    plt.axvline(x=np.mean(flat_sims), color='orange', linestyle='--', linewidth=2, label=f'Mittelwert ({np.mean(flat_sims):.3f})')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()